# S1 2024 COMPSCI 714 - Tutorial 5: Convolutions, Convolutional Neural Networks (CNNs) and Vision Transformer (ViT)

Welcome to Tutorial 5! This tutorial covers basics of convolutions, implementing CNNs, using pre-trained CNNs, using a 1-D CNN to predict time series and using a Vision Transformer. 

Disclaimer/Copyright: some parts of the code and text used in this Notebook is directly reused or adapted from Aurélien Géron's notebooks https://github.com/ageron/handson-ml3/blob/main/14_deep_computer_vision_with_cnns.ipynb, https://github.com/ageron/handson-ml3/blob/main/15_processing_sequences_using_rnns_and_cnns.ipynb and his book "Hands-on Machine Learning with Scikit-Learn, Keras and Tensorflow, Ed.3", more particularly from Chapters 14 and 15. 

Most of this tutorial uses TensorFlow/Keras, but links will be provided to related content in PyTorch.

In [ ]:
import sys
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

In [ ]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
if not tf.config.list_physical_devices('GPU'):
    print("No GPU was detected. Neural nets can be very slow without a GPU.")
    if "google.colab" in sys.modules:
        print("Go to Runtime > Change runtime and select a GPU hardware "
              "accelerator.")

It is recommended to run this Notebook on a GPU (e.g., on Colab) to accelarate the training of the CNN model in Part II.

## I. CNN layers

### I.1. Convolutional Layers

*Convolutional layers* are the basis of *Convolutional Neural Networks* (CNNs). In a CNN, convolutional layers take an image or feature map as input, and output a feature map (i.e., a filtered or compressed version of the input image). One important difference between convolutional layers and fully-connected layers is the way they connect inputs and outputs.  
In a fully-connected layer, each input is connected to each output. In other words, every neuron in one layer is connected to every single neuron in the next layer.  
In a convolutional layer, each input (i.e., pixel) is connected only to a specific *field* of outputs, defined by the filter size associated to this layer. In other words, each neuron in a layer is only connected to neurons located in a small rectangle of the previous layer. 

We will first apply a convolutional layer on an image to see what effect it has. 

Let's first load sample images from the `sklearn.datasets` module and display their shape.

In [ ]:
from sklearn.datasets import load_sample_images
import tensorflow as tf

images = load_sample_images()["images"]
images = tf.keras.layers.CenterCrop(height=70, width=120)(images)
images = tf.keras.layers.Rescaling(scale=1 / 255)(images)

In [ ]:
images.shape

**TODO**: Analyse the output of the previous cell and answer the following questions:
- How many images are there in this tensor?
- What is the height and width of each image?
- How many channels are the images encoded with?

Once you are done, you can run the following cell. 

In [ ]:
def read_insights(file):
    with open(file, "r") as file:
        print(file.read())
read_insights("do_not_read.txt")

You can display the images in grayscale with the following cell.

In [ ]:
plt.figure(figsize=(15, 9))
for image_idx in (0, 1):
    for fmap_idx in (0, 1):
        plt.subplot(2, 2, image_idx * 2 + fmap_idx + 1)
        plt.imshow(images[image_idx, :, :, fmap_idx], cmap="gray")
        plt.axis("off")

plt.show()

#### I.1.i. Running convolutions

With Keras/TensorFlow, you can use directly the `Conv2D` layer to perform a convolution on an image.  
Note that the equivalent in PyTorch is also called the [`Conv2D` layer](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html).  
You might ask yourself, why adding *2D*? This is because you can perform convolutions with different number of dimensions, e.g. we will use the 1D convolution a little bit later in this tutorial. The `Conv2D` layer specifically performs convolution in 2 dimensions. 

Let's apply a convolution layer with 16 filters and a kernel size of 7 to the images we loaded just before.

In [ ]:
conv_layer = tf.keras.layers.Conv2D(filters=16, kernel_size=7)
fmaps = conv_layer(images)

In [ ]:
fmaps.shape

Notice that the images shrunked a little bit in height and width! This is an effect of the convolution process, as discussed in the lectures.  
The tensor now contains 16 channels (i.e., feature maps), 1 for each filter applied to the image. Each filter was initialised randomly in the layer, so the each of the 16 output feature maps is the result of filtering with a random filter. You can visualise the 2 first output feature maps with the code below. 

In [ ]:
plt.figure(figsize=(15, 9))
for image_idx in (0, 1):
    for fmap_idx in (0, 1):
        plt.subplot(2, 2, image_idx * 2 + fmap_idx + 1)
        plt.imshow(fmaps[image_idx, :, :, fmap_idx], cmap="gray")
        plt.axis("off")

plt.show()

As you can see, randomly generated filters typically act like general edge detectors. Edge detectors are useful tools in image processing, and this is what the filters in a convolutional layer start as. Then, during training, each filter will become more specialised to recognise useful patterns for the task.

#### I.1.ii. Padding

What if we want to avoid the image to shrink during the convolution process? Well, we can use *padding* (i.e., add extra rows and columns of zeroes around the image). Remember that you have the choice between two options:
- Valid padding: equivalent to using no padding.
- Same padding: adding padding such as the output feature map dimensions are the same as the input.

You just need to set the `padding` parameter to `"valid"` or `"same"` in the `Conv2D` layer. It is set to `"valid"` by default.

In [ ]:
conv_layer = tf.keras.layers.Conv2D(filters=16, kernel_size=7, padding="same")
fmaps = conv_layer(images)

In [ ]:
fmaps.shape

Notice that the output height and width are the same as the input with `padding="same"`.

#### I.1.iii. Strided convolution

To reduce the computational cost of convolutions, it is possible to skip some convolution steps (i.e., reducing the number of connections from one layer to the next in a network). This is called *strided convolution*.  
You just need to set the `strides` parameter to a value different from 1 (the default value).

In [ ]:
conv_layer = tf.keras.layers.Conv2D(filters=16, kernel_size=7, padding="same", strides=2)
fmaps = conv_layer(images)

In [ ]:
fmaps.shape

Notice that the output feature map is shrunked when setting `strides` to a number greater than 1, even if `padding` is set to `"same"`. In that case, the padding is calculated such as $o = ceil(i/s)$ (with $o$ the output feature map size and $i$ the input image/feature size). If the input image height is not equal to its width, then this is applied separately for each dimension.

### I.2. Pooling layers

CNNs are also using another type of layers called *pooling layers*. These layers are used to compress the information in the network. They *subsample* the image to reduce the computational load, memory usage and number of parameters. They can also be seen as selecting relevant information to pass forward in the network. They work similarly to convolutional layers, where each neuron in a layer is connected to a limited number of neurons in the next layer, but they do not filther the image. They calculate a selected statistics (e.g., mean or max) on patches of the input image/feature map and propagate these values forward to the next layer.

The two main poolings which are used in CNNs are max pooling (i.e., selecting the max pixel value of the patch) and average pooling (i.e., selecting the average pixel value of the patch). With TensorFlow/Keras, you can use the `MaxPool2D`layer and `AveragePooling2D` layer for applying max and average pooling. The `pool_size` parameter is used to set the size of the patch used for pooling.  
There exists similar layers with the same names (but different parameters' names) in [PyTorch](https://pytorch.org/docs/stable/nn.html#pooling-layers). 

In [ ]:
max_pool = tf.keras.layers.MaxPool2D(pool_size=2)
output = max_pool(images)

In [ ]:
import matplotlib as mpl

fig = plt.figure(figsize=(12, 8))
gs = mpl.gridspec.GridSpec(nrows=1, ncols=2, width_ratios=[2, 1])

ax1 = fig.add_subplot(gs[0, 0])
ax1.set_title("Input")
ax1.imshow(images[0])  # plot the 1st image
ax1.axis("off")
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_title("Output")
ax2.imshow(output[0])  # plot the output for the 1st image
ax2.axis("off")
plt.show()

**TODO**: Try different `pool_size` values and replace `MaxPool2D` by `AveragePooling2D` which performs average pooling. Discuss the different results. 

Let's put some of these layers together to form a CNN!

## II. CNNs with Keras/Tensorflow

### II.1. Building and training a CNN
We will reuse the Fashion MNIST dataset we used in a previous tutorial.

In [ ]:
f_mnist = tf.keras.datasets.fashion_mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = f_mnist
X_train_full = np.expand_dims(X_train_full, axis=-1).astype(np.float32) / 255
X_test = np.expand_dims(X_test.astype(np.float32), axis=-1) / 255
X_train, X_valid = X_train_full[:-5000], X_train_full[-5000:]
y_train, y_valid = y_train_full[:-5000], y_train_full[-5000:]

Let's now create a CNN with Keras/TensorFlow! You should be familiar with the Sequential API now. The principle is the same as with the other types of architecture we described in previous tutorials. You can add layers sequentially to build the model. Note that we first create a default configuration for the convolutional layers we will use (kernel size of 3, same padding, ReLU activation and He normal initialisation), called `DefaultConv2D`, using the `partial` function introduced in Tutorial 3. 

In [ ]:
from functools import partial

DefaultConv2D = partial(tf.keras.layers.Conv2D, kernel_size=3, padding="same",
                        activation="relu", kernel_initializer="he_normal")
model = tf.keras.Sequential([
    DefaultConv2D(filters=64, kernel_size=7, input_shape=[28, 28, 1]),
    tf.keras.layers.MaxPool2D(),
    DefaultConv2D(filters=128),
    DefaultConv2D(filters=128),
    tf.keras.layers.MaxPool2D(),
    DefaultConv2D(filters=256),
    DefaultConv2D(filters=256),
    tf.keras.layers.MaxPool2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=128, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(units=64, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(units=10, activation="softmax")
])

**TODO**: Identify the different components of this CNN. How many layers does it have? What types of layers? Do you recognise any other components we discussed in previous tutorials?

Once you are done, you can run the following cell to confirm your analysis.

In [ ]:
def read_insights(file):
    with open(file, "r") as file:
        print(file.read())
read_insights("do_not_read2.txt")

You can now compile this model and train it on the train set.

In [ ]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam",
              metrics=["accuracy"])

**WARNING**: It is recommended to run the following cell on a GPU (e.g., on Colab). The model will take only a few minutes to train on a GPU, while it might take 10 to 20 min on a CPU. 

In [ ]:
history = model.fit(X_train, y_train, epochs=10,
                    validation_data=(X_valid, y_valid))

If you are happy with the validation performance, you can test the model on the test set, and use it to predict new images.

In [ ]:
score = model.evaluate(X_test, y_test)
X_new = X_test[:10]  # pretend we have new images
y_pred = model.predict(X_new)

Congrats, you have train and tested your (maybe) first CNN!

**TODO (later)**: After the tutorial session, you can come back to this part and try to implement the classical LeNet-5, Alexnet, and/or VGGNet architectures. The GoogleNet and ResNet implement more specify blocks, but you can also have a go at them if you feel like trying. 

Else, we will next how to load pre-trained CNNs to use them directly without having to implement and train them.

**PyTorch**: If want to see how you can define and train a CNN with PyTorch, you can have a look at the [Training a Classifer PyTorch tutorial on the Cifar10 datatset](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html).

### II.2. Using pre-trained CNNs from Keras/TensorFlow

Most standards CNNs models like GoogLeNet or ResNet are readily available in pre-trained versions (i.e., you do not have to implement and train them by yourself again). Of course, you can try if you would like to and if you have computational resources to spare. You will see later in the semester that it is also possible to only retrain part of a pre-trained network on a target task, using transfer learning. For now, we will focus on loading pre-trained models and using them, as we did in Tutorial 4 with the Huggin Face Transformer library. This time, we will use the `tf.keras.applications` package from which you can load a [range of pre-trained standard CNNs] (https://www.tensorflow.org/api_docs/python/tf/keras/applications), such as versions of ResNet and VGGNet.

For example, let's load the ResNet-50 model (50 layers). You can do this with 1 line of code and load the pre-trained model within a few seconds! It would take days or weeks to train, depending on the hardware you use.

In [ ]:
model = tf.keras.applications.ResNet50(weights="imagenet")

The parameter `weights="imagenet"` specifies that we load the weights pre-trained on the ImageNet dataset, which contains 1 to several millions images depending on the version, and [1000 classes](https://observablehq.com/@mbostock/imagenet-hierarchy). The task it was trained for is to classify an input image into these classes. 

A ResNet-50 model takes input images with spatial dimensions 224 $\times$ 224 (this might vary depending on the model you load). Keras provides a `Resizing` layer you can use to resize an image you want to input to the model for prediction. This layer will first crop the image to the target aspect ratio before applying the resizing. Let's reuse the sample images used in Part I.

In [ ]:
images = load_sample_images()["images"]
images_resized = tf.keras.layers.Resizing(height=224, width=224,
                                          crop_to_aspect_ratio=True)(images)

The pre-trained model expects images preprocessed in a specific way. E.g., somtimes they expect pixel values scaled between 0 and 1, or -1 and 1. Each model from the `tf.keras.applications` package provides a `preprocess_input()` function that you can use to preprocess images for this specific model. Note that these functions assume that the original images have pixel values between 0 and 255.

In [ ]:
inputs = tf.keras.applications.resnet50.preprocess_input(images_resized)

You can now use the model to make predictions on the two sample images. 

In [ ]:
Y_proba = model.predict(inputs)
Y_proba.shape # 2 predictions, with each 1000 outputs (proba for each class)

Out of the 1000 predicted probabilities (1 for each class), you can select the top *K*, with the class names and associated probabilities. To do so, you can use the `decode_predictions()` function. For each image, it returns an array containing the top *K* predictions, with the corresponding [ImageNet/WordNet class identifier](https://gist.github.com/aaronpolhamus/964a4411c0906315deb9f4a3723aac57), class name and confidence score (i.e., probability from softmax). 

In [ ]:
top_K = tf.keras.applications.resnet50.decode_predictions(Y_proba, top=3)
for image_index in range(len(images)):
    print(f"Image #{image_index}")
    for class_id, name, y_proba in top_K[image_index]:
        print(f"  {class_id} - {name:12s} {y_proba:.2%}")

The true labels for Image #0 and Image #1 are respectively palace and dahlia. The prediction for the first image is correct, but not for the second one. This is explained by the fact that the ImageNet dataset does not contain any dahlia class. The model then interprets the image as a vase, daisy or maybe honeycomb, with relatively low confidence (compared to the first image at least). Daisy is a reasonable guess given that this is another type of flower from the same family (Compositea). Honeycomb also makes sense if you look at the structure of the flower (see below). Vase is more enigmatic (to me at least). Maybe the shape looks like some vases images the model trained on, or maybe as flowers are often in vases, so associated somehow with them in the model. The interpretation is up to you!

In [ ]:
plt.figure(figsize=(10, 6))
for idx in (0, 1):
    plt.subplot(1, 2, idx + 1)
    plt.imshow(images_resized[idx] / 255)
    plt.axis("off")

plt.show()

**Todo**: You can try to predict any image of your choice. You can use the `tf.keras.utils.load_img()` function to load an image at a particular path. After loading the image, you need to use the `tf.expand_dims()` function with parameter `axis=0` to add a fourth dimention to the image tensor, because the `model.predict()` function expects several images as input. An example is shown below.  
You can also try to load an play with other models from the `tf.keras.applications` package. 

The following example is making a prediction for a single image, loaded from the images folder. 

In [ ]:
img = tf.keras.utils.load_img("images/Heron.png")
img_resized = tf.keras.layers.Resizing(height=224, width=224,
                                          crop_to_aspect_ratio=True)(img)

# Display the image
plt.imshow(img_resized / 255)
plt.axis("off")
plt.show()

In [ ]:
img_resized = tf.expand_dims(img_resized, axis=0)
input = tf.keras.applications.resnet50.preprocess_input(img_resized)
Y_proba = model.predict(input)
top_K = tf.keras.applications.resnet50.decode_predictions(Y_proba, top=3)

In [ ]:
for class_id, name, y_proba in top_K[0]:
        print(f"  {class_id} - {name:12s} {y_proba:.2%}")

The image predicted is actually a white faced heron I photographed a few years back on the border of lake Waikaremoana. Knowing that this exact type of heron is not a class in the ImageNet dataset, the predictions are not bad! I.e., a crane is a bird looking quite similar to a heron. 

## III. 1D CNN-GRU network and Wavenet for time series

1D convolution slides a 1D kernel over a sequence of values. Similarly to a 2D convolution layer, a 1D convolution layer slides several kernels across a sequence, producing a 1D feature map per kernel. When training the layer, each kernel will learn to detect a short sequential pattern, limited by the kernel size. 

### III.1. 1D CNN-GRU network for predicting time series

#### III.1.i. Time series dataset and task

For this part, we will reuse the train/rail ridership dataset we used in Tutorial 4. However, we will modify the task slightly. Instead of just predicting the next day rail ridership, we will predict the next 14 days at once. 

Let's load the data first.

In [ ]:
tf.keras.utils.get_file(
    "ridership.tgz",
    "https://github.com/ageron/data/raw/main/ridership.tgz",
    cache_dir=".",
    extract=True
)
path = Path("datasets/ridership/CTA_-_Ridership_-_Daily_Boarding_Totals.csv")
df = pd.read_csv(path, parse_dates=["service_date"])
df.columns = ["date", "day_type", "bus", "rail", "total"]  # shorter names
df = df.sort_values("date").set_index("date")
df = df.drop("total", axis=1)  # no need for total, it's just bus + rail
df = df.drop_duplicates()  # remove duplicated months (2011-10 and 2014-07)

We will use multivariate data to make predictions (i.e., both bus and train data + the next day type). Run the next cell to build the multivariate dataset and split it into train, validation and test periods (same as in part I.3.iv. of Tutorial 4). 

In [ ]:
df_mulvar = df[["bus", "rail"]] / 1e6  # use both bus & rail series as input
df_mulvar["next_day_type"] = df["day_type"].shift(-1)  # we know tomorrow's type
df_mulvar = pd.get_dummies(df_mulvar, dtype = np.float32)  # one-hot encode the day type

mulvar_train = df_mulvar["2016-01":"2018-12"]
mulvar_valid = df_mulvar["2019-01":"2019-05"]
mulvar_test = df_mulvar["2019-06":]

To split the data into input sequences and targets for the sepcific task (i.e., predicting the next 14 days at once), we use the `to_windows()` and `to_seq2seq_dataset()` helper functions below. They perform equivalent operations than the `timeseries_dataset_from_array()` function you used in Tutorial 4, but this time creating windows of `seq_length` length as input and windows of 14 consecutive elements as targets.  
We will not detail how these fucntions work more in details here, but you can read about it in Chapter 15 (pp.552-554, 563) of the "Hands-on ML book".

In [ ]:
def to_windows(dataset, length):
    dataset = dataset.window(length, shift=1, drop_remainder=True)
    return dataset.flat_map(lambda window_ds: window_ds.batch(length))
    
def to_seq2seq_dataset(series, seq_length=56, ahead=14, target_col=1,
                       batch_size=32, shuffle=False, seed=None):
    ds = to_windows(tf.data.Dataset.from_tensor_slices(series), ahead + 1)
    ds = to_windows(ds, seq_length).map(lambda S: (S[:, 0], S[:, 1:, 1]))
    if shuffle:
        ds = ds.shuffle(8 * batch_size, seed=seed)
    return ds.batch(batch_size)

#### III.1.ii. 1D CNN-GRU model
Let's now build the network architecture. We will combine a 1D convolution layer with 32 filters, a kernel size of 4 and a stride of 2 with a GRU layer, followed by a fully connected output layer with 14 neurons (1 for each day we predict). To use 1D convolution in Keras\TensorFlow, you can use the `Conv1D` layer. There exists a similar layer with the same name in [PyTorch](https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html).

In [ ]:
conv_rnn_model = tf.keras.Sequential([
    tf.keras.layers.Conv1D(filters=32, kernel_size=4, strides=2,
                           activation="relu", input_shape=[None, 5]),
    tf.keras.layers.GRU(32, return_sequences=True),
    tf.keras.layers.Dense(14)
])

Because we use a stride of 2, the 1D convolution layer will downsample the input sequence by a factor 2 (remember that striding shrinkes the output feature map). The convolution process will help the network extract (hopefully) useful information from the input sequence by creating a range of output feature maps resulting from the filtering of the input by the different filters.  
Because the convolution layer is shortening the sequences, it may help the GRU layer detecting longer patterns. Therefore, we can increase the input sequence length to 112 days (we kept it at 56 day in Tutorial 4).  
Note that we also need to remove the 3 first time steps in the targets as the kernel size is 4 (i.e., the first output of the convolution layer will be based on input time steps 0 to 3 (inclusive), and the first forecast will be for time steps 4 to 17 (instead of 1 to 14).  
We also need to downsample the targets by a factor 2 (because of the stride). 

All of this is done in the next cell.

In [ ]:
longer_train = to_seq2seq_dataset(mulvar_train, seq_length=112,
                                       shuffle=True, seed=42)
longer_valid = to_seq2seq_dataset(mulvar_valid, seq_length=112)
downsampled_train = longer_train.map(lambda X, Y: (X, Y[:, 3::2]))
downsampled_valid = longer_valid.map(lambda X, Y: (X, Y[:, 3::2]))

Finally, you can train and evaluate the model by using the same `fit_and_evaluate()` helper fucntion we used in Tutorial 4. 

In [ ]:
def fit_and_evaluate(model, train_set, valid_set, learning_rate, epochs=500):
    early_stopping_cb = tf.keras.callbacks.EarlyStopping(
        monitor="val_mae", patience=50, restore_best_weights=True)
    opt = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    model.compile(loss=tf.keras.losses.Huber(), optimizer=opt, metrics=["mae"])
    history = model.fit(train_set, validation_data=valid_set, epochs=epochs,
                        callbacks=[early_stopping_cb])
    valid_loss, valid_mae = model.evaluate(valid_set)
    return valid_mae * 1e6 # Reverse the scaling 

In [ ]:
fit_and_evaluate(conv_rnn_model, downsampled_train, downsampled_valid,
                 learning_rate=0.1, epochs=10)

**TODO**: Define and train a network with only 1 hidden GRU layer, and 1 output layer for the same task. Compare the performance with the CNN-GRU network you just trained.

### III.2. WaveNet (stacked 1D convolution layers)

It is also possible use network architectures only composed of stacked 1D convolution layers to work with sequential data. This is the case for architecture of the WaveNet model we discussed in the lecture. 

We will use a simplified implementation of the WaveNet architecture in this tutorial, but you can find the original implementation (as defined in the article) in the [Chapter 15's Notebook of the "Hands-on ML" book](https://github.com/ageron/handson-ml3/blob/main/15_processing_sequences_using_rnns_and_cnns.ipynb) (look for "Extra Material – Wavenet Implementation").  
You can find the simplified WaveNet implementation in the next cell. 

In [ ]:
wavenet_model = tf.keras.Sequential()
wavenet_model.add(tf.keras.layers.InputLayer(input_shape=[None, 5]))
for rate in (1, 2, 4, 8) * 2:
    wavenet_model.add(tf.keras.layers.Conv1D(
        filters=32, kernel_size=2, padding="causal", activation="relu",
        dilation_rate=rate))
wavenet_model.add(tf.keras.layers.Conv1D(filters=14, kernel_size=1))

Let's analyse the structure step by step:
- It starts with an explicit input layer. This is because we use a for loop to add `Conv1D` layer sequentially next. Therefore, it is easier to explicitely use an `InputLayer` layer to set the `input_shape`, instead of trying to do it only for the first hidden layer in the for loop.
- Then, several `Conv1D` layers are added, with `causal` padding. This is the same as `same` padding except that zeros are added only at the start of the input sequence, instead of on both sides. This ensures the convolution layer does not look into the future when making predictions. Each layer is added with a different `dilation_rate` value. We add 2 times 4 layers with growing dilation rates: 1, 2, 4, 8, and again.
- Finally, the network ends with a `Conv1D` output layer with 14 filters, a kernel size of 1 and not activation function. This is actually equivalent to a fully connected layer (`Dense` layer) with 14 neurons!

Because `causal` padding is used in the hidden layers, each of them outputs a sequence of the same length as the input sequence. Therefore, the targets we are using can be the full 112-day sequences. No need to remove any time steps or to downsample them here. 

You can now train and evaluate this model on the 14-day rail ridership prediction task.

In [ ]:
fit_and_evaluate(wavenet_model, longer_train, longer_valid,
                 learning_rate=0.1, epochs=10)

You should see that the performance is close to what you got with the 1D CNN-GRU model previously. However, depending on the task, this architecture can achieve SOTA performance, e.g., on various audio tasks. Some examples of voices or music audio samples generated by the WaveNet model trained by DeepMind are available on the [Google DeepMind WaveNet blog article](https://deepmind.google/discover/blog/wavenet-a-generative-model-for-raw-audio/).

## IV. Vision Transformer VS ResNet using Hugging Face

In this last part of the Tutorial, you will use pre-trained versions of the Vision Transformer model and ResNet (CNN based) models to classify images. Both models have been trained on the ImageNet dataset, just as the ResNet-50 model we used in section II.2.
We will not go over the implementation of the Vision Transformer architecture, but you can have a look at the following tutorials to do it in Keras/TensorFlow and PyTorch:
- [Image classification with Vision Transformer (Keras/TensorFlow)](https://keras.io/examples/vision/image_classification_with_vision_transformer/)
- [UVADCL Tutorial: Vision Transformers (PyTorch)](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial15/Vision_Transformer.html)

Some of the code in this part might raise depreciation warnings. If you want to filter them (they are not really useful here), you can run the following cell. 

In [ ]:
import warnings
warnings.filterwarnings('error', category=DeprecationWarning)

Next, let's import the `TFViTForImageClassification`, `TFResNetForImageClassification` and `AutoImageProcessor` modules from the  Hugging Face `transformers` package. They contain respective classes and functions to use ViT and ResNet models for classification, and dedicated image processors (to preprocess data for each type of model, similarly as in part II.2. with ResNet-50).

In [ ]:
from transformers import AutoImageProcessor, TFViTForImageClassification, TFResNetForImageClassification

Next, we will also import the Hugging Face `datasets` package. This package gives you access to the [Hugging Face Datasets library](https://huggingface.co/docs/datasets/en/index), which allows you to easily access and share a range of datasets for Audio, Computer Vision, and Natural Language Processing (NLP) tasks.

**Important**: You might need to install the `datasets` package with pip in your virtual environment (or globally if you are not using one) first. This package was not listed in the requirement file (sorry!). 

In [ ]:
from datasets import load_dataset

Let's load an image from the Hugging Face [`huggingface/cats-image` repository](https://huggingface.co/datasets/huggingface/cats-image) to test the models after we load them.

In [ ]:
dataset = load_dataset("huggingface/cats-image")
image = dataset["test"]["image"][0]

Run the following cell to display the image. 

In [ ]:
plt.imshow(image)
plt.axis("off")
plt.show()

Let's now load the ViT model and associated processor. 

In [ ]:
model_ViT = TFViTForImageClassification.from_pretrained("google/vit-base-patch16-224")
image_processor_ViT = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")

**TODO**: Before running the following cell, analyse briefly what each part of the code is doing. 

In [ ]:
inputs = image_processor_ViT(image, return_tensors="tf")
logits = model_ViT(**inputs).logits

# model predicts one of the 1000 ImageNet classes
predicted_label = int(tf.math.argmax(logits, axis=-1))
print(model.config.id2label[predicted_label])

The image is first preprocessed using the image processor dedicated to this version of the ViT model. Then, a prediction is made with the image we loaded previously. An argmax function is then applied to the output of the model (logits). The class with the highest probability is therefore displayed.  

The model classifies the image as in the "Egyptian cat" category, which is one of the 1000 possible classes of the ImageNet dataset. There is indeed a cat on the image (even 2!), but they do not really look like an Egyptian breed. Nervertheless, this is a very good guess out of 1000 classes!

Let see now what ResNet predicts!

We do the exact same process with the pre-trained Resnet-50 model loaded from Hugging Face this time.

In [ ]:
image_processor_ResNet = AutoImageProcessor.from_pretrained("microsoft/resnet-50")
model_ResNet = TFResNetForImageClassification.from_pretrained("microsoft/resnet-50")

In [ ]:
inputs = image_processor_ResNet(image, return_tensors="tf")
logits = model_ResNet(**inputs).logits

# model predicts one of the 1000 ImageNet classes
predicted_label = int(tf.math.argmax(logits, axis=-1))
print(model.config.id2label[predicted_label])

The ResNet model classifies the image in the "tiger cat" category. This is also good given the striped pattern exhibited by the cats on the images. 

**To continue**: Feel free to play with different part of the code in this tutorial, or just load more images and classify them with the different pre-trained models you loaded in part II.2. and IV. 